# UNIVERSITY OF CAPE COAST
# College of Humanities and Legal Studies
# School of Economics

# DMA 820 — Data Curation and Management
## Group 1: Macroeconomic time-series data (Sub-Saharan Africa)

## Script: data_preparation.py
### Purpose: Clean, reshape, validate, and export the WDI extract.
### Input: data/raw/P_Data_Extract_From_World_Development_Indicators.xlsx
### Output: data/processed/curated_dataset.csv


In [1]:
import pandas as pd
import numpy as np
import re
import os

print("python:", pd.__version__)
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

python: 2.2.3
pandas: 2.2.3
numpy : 2.1.3


In [2]:
# Define project-relative paths
RAW_PATH = "P_Data_Extract_From_World_Development_Indicators.xlsx"
OUT_PATH = "data/processed/curated_dataset.csv"

# Create output folder if it does not exist
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

print("Raw file exists:", os.path.exists(RAW_PATH))

Raw file exists: True


In [3]:
# Load raw data
# The 'Data' sheet contains the observations.
# The last rows of the sheet contain footers ("Data from database...",
# "Last Updated..."), so we skip them.

#RAW = r"C:\Users\user\Desktop\ANACONDA\DATA CURATION\P_Data_Extract_From_World_Development_Indicators.xlsx"
#df = pd.read_excel(RAW, sheet_name="Data", skipfooter=5)

# raw = pd.read_excel(RAW)

raw = pd.read_excel(
    RAW_PATH,
    sheet_name="Data",
    skipfooter=5,      # drops the WDI footer rows
    engine="openpyxl"
)


print("Raw shape:", raw.shape)
raw.head()

Raw shape: (960, 12)


,Country Name,Country Code,Time,Time Code,GDP growth (annual %) [NY.GDP.MKTP.KD.ZG],GDP per capita (constant 2015 US$) [NY.GDP.PCAP.KD],"Inflation, consumer prices (annual %) [FP.CPI.TOTL.ZG]","Unemployment, total (% of total labor force) (modeled ILO estimate) [SL.UEM.TOTL.ZS]",General government final consumption expenditure (% of GDP) [NE.CON.GOVT.ZS],Trade (% of GDP) [NE.TRD.GNFS.ZS],"Foreign direct investment, net inflows (% of GDP) [BX.KLT.DINV.WD.GD.ZS]",Gross capital formation (% of GDP) [NE.GDI.TOTL.ZS]
0,Angola,AGO,2006,YR2006,11.841818,3065.055762,13.30521,16.123,16.547943,84.624604,-0.064301,24.128522
1,Angola,AGO,2007,YR2007,13.002689,3336.363325,12.251497,16.066,16.768658,97.225631,-1.223123,26.872321
2,Angola,AGO,2008,YR2008,10.786636,3559.195225,12.475829,16.152,18.201028,108.680409,1.699528,31.047664
3,Angola,AGO,2009,YR2009,1.995589,3494.807245,13.730284,16.383,20.680618,102.263183,2.699092,42.112663
4,Angola,AGO,2010,YR2010,5.293666,3540.791596,14.469656,16.595,18.052816,90.994063,-3.377619,29.763833


In [4]:
# Rename columns to machine-readable names ---
rename_map = {
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Time": "year",
    "Time Code": "time_code",
}
# Extract indicator codes from header brackets
for col in raw.columns:
    m = re.search(r"\[([^\]]+)\]", str(col))
    if m:
        rename_map[col] = m.group(1)
raw = raw.rename(columns=rename_map)

print("Columns after rename:")
print(list(raw.columns))

Columns after rename:
['country_name', 'country_code', 'year', 'time_code', 'NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.KD', 'FP.CPI.TOTL.ZG', 'SL.UEM.TOTL.ZS', 'NE.CON.GOVT.ZS', 'NE.TRD.GNFS.ZS', 'BX.KLT.DINV.WD.GD.ZS', 'NE.GDI.TOTL.ZS']


In [5]:
# Drop the time_code column (redundant) ---
raw = raw.drop(columns=["time_code"], errors="ignore")
raw.head()

,country_name,country_code,year,NY.GDP.MKTP.KD.ZG,NY.GDP.PCAP.KD,FP.CPI.TOTL.ZG,SL.UEM.TOTL.ZS,NE.CON.GOVT.ZS,NE.TRD.GNFS.ZS,BX.KLT.DINV.WD.GD.ZS,NE.GDI.TOTL.ZS
0,Angola,AGO,2006,11.841818,3065.055762,13.30521,16.123,16.547943,84.624604,-0.064301,24.128522
1,Angola,AGO,2007,13.002689,3336.363325,12.251497,16.066,16.768658,97.225631,-1.223123,26.872321
2,Angola,AGO,2008,10.786636,3559.195225,12.475829,16.152,18.201028,108.680409,1.699528,31.047664
3,Angola,AGO,2009,1.995589,3494.807245,13.730284,16.383,20.680618,102.263183,2.699092,42.112663
4,Angola,AGO,2010,5.293666,3540.791596,14.469656,16.595,18.052816,90.994063,-3.377619,29.763833


In [6]:
raw.shape

(960, 11)

In [7]:
# Identifying the Indicator Columns
# Indicator codes follow the WDI pattern:  XX.XXX.XXXX.XX.XX
indicator_cols = [c for c in raw.columns if re.match(r"^[A-Z]{2}\.", c)]

print("Indicators found:")
for c in indicator_cols:
    print(" -", c)

Indicators found:
 - NY.GDP.MKTP.KD.ZG
 - NY.GDP.PCAP.KD
 - FP.CPI.TOTL.ZG
 - SL.UEM.TOTL.ZS
 - NE.CON.GOVT.ZS
 - NE.TRD.GNFS.ZS
 - BX.KLT.DINV.WD.GD.ZS
 - NE.GDI.TOTL.ZS


In [8]:
# Replace WDI missing marker ".." with NA ---
raw = raw.replace("..", np.nan)

# Coerce indicator columns to numeric
for c in indicator_cols:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# Quick check
print(raw[indicator_cols].isna().sum())

NY.GDP.MKTP.KD.ZG        27
NY.GDP.PCAP.KD           26
FP.CPI.TOTL.ZG           72
SL.UEM.TOTL.ZS           25
NE.CON.GOVT.ZS          127
NE.TRD.GNFS.ZS          124
BX.KLT.DINV.WD.GD.ZS     70
NE.GDI.TOTL.ZS          125
dtype: int64


C:\Users\user\AppData\Local\Temp\ipykernel_1744\19980747.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw = raw.replace("..", np.nan)


In [9]:
# Reshape wide → long ---
indicator_cols = [c for c in raw.columns if re.match(r"^[A-Z]{2}\.", c)]
long_df = raw.melt(
    id_vars=["country_name", "country_code", "year"],
    value_vars=indicator_cols,
    var_name="indicator_code",
    value_name="value",
)

print("Long shape:", long_df.shape)
long_df.head()

Long shape: (7680, 5)


,country_name,country_code,year,indicator_code,value
0,Angola,AGO,2006,NY.GDP.MKTP.KD.ZG,11.841818
1,Angola,AGO,2007,NY.GDP.MKTP.KD.ZG,13.002689
2,Angola,AGO,2008,NY.GDP.MKTP.KD.ZG,10.786636
3,Angola,AGO,2009,NY.GDP.MKTP.KD.ZG,1.995589
4,Angola,AGO,2010,NY.GDP.MKTP.KD.ZG,5.293666


In [10]:
# 6 Indicator metadata taken from the WDI "Series - Metadata" sheet.

indicator_meta = {
    "NY.GDP.MKTP.KD.ZG": {
        "indicator_name": "GDP growth (annual %)",
        "unit": "percent",
        "price_basis": "constant price",
        "observation_status": "reported",
    },
    "NY.GDP.PCAP.KD": {
        "indicator_name": "GDP per capita (constant 2015 US$)",
        "unit": "constant 2015 US$",
        "price_basis": "constant price",
        "observation_status": "reported",
    },
    "FP.CPI.TOTL.ZG": {
        "indicator_name": "Inflation, consumer prices (annual %)",
        "unit": "percent",
        "price_basis": "current price",
        "observation_status": "reported",
    },
    "SL.UEM.TOTL.ZS": {
        "indicator_name": "Unemployment, total (% of labour force, modelled ILO)",
        "unit": "percent",
        "price_basis": "modelled",
        "observation_status": "modelled",
    },
    "NE.CON.GOVT.ZS": {
        "indicator_name": "General government final consumption expenditure (% of GDP)",
        "unit": "percent",
        "price_basis": "current price",
        "observation_status": "reported",
    },
    "NE.TRD.GNFS.ZS": {
        "indicator_name": "Trade (% of GDP)",
        "unit": "percent",
        "price_basis": "current price",
        "observation_status": "reported",
    },
    "BX.KLT.DINV.WD.GD.ZS": {
        "indicator_name": "Foreign direct investment, net inflows (% of GDP)",
        "unit": "percent",
        "price_basis": "current price",
        "observation_status": "reported",
    },
    "NE.GDI.TOTL.ZS": {
        "indicator_name": "Gross capital formation (% of GDP)",
        "unit": "percent",
        "price_basis": "current price",
        "observation_status": "reported",
    },
}

long_df["indicator_name"]     = long_df["indicator_code"].map(lambda x: indicator_meta[x]["indicator_name"])
long_df["unit"]               = long_df["indicator_code"].map(lambda x: indicator_meta[x]["unit"])
long_df["price_basis"]        = long_df["indicator_code"].map(lambda x: indicator_meta[x]["price_basis"])
long_df["observation_status"] = long_df["indicator_code"].map(lambda x: indicator_meta[x]["observation_status"])

long_df.head()

,country_name,country_code,year,indicator_code,value,indicator_name,unit,price_basis,observation_status
0,Angola,AGO,2006,NY.GDP.MKTP.KD.ZG,11.841818,GDP growth (annual %),percent,constant price,reported
1,Angola,AGO,2007,NY.GDP.MKTP.KD.ZG,13.002689,GDP growth (annual %),percent,constant price,reported
2,Angola,AGO,2008,NY.GDP.MKTP.KD.ZG,10.786636,GDP growth (annual %),percent,constant price,reported
3,Angola,AGO,2009,NY.GDP.MKTP.KD.ZG,1.995589,GDP growth (annual %),percent,constant price,reported
4,Angola,AGO,2010,NY.GDP.MKTP.KD.ZG,5.293666,GDP growth (annual %),percent,constant price,reported


In [11]:
# Distinguish Missing from Zero
# Rule:
#   value = NaN    -> missing
#   value = 0      -> true zero
#   otherwise      -> observed

# Flag missing vs zero ---
long_df["value_status"] = np.where(
    long_df["value"].isna(), "missing",
    np.where(long_df["value"] == 0, "zero", "observed")
)

print(long_df["value_status"].value_counts())

value_status
observed    7079
missing      596
zero           5
Name: count, dtype: int64


In [12]:
# Adding Constant Metadata Fields
# date shown in the WDI export
long_df["periodicity"]   = "Annual"
long_df["source_dataset"] = "World Development Indicators"
long_df["retrieval_date"] = "2026-07-13"   

In [13]:
# Duplicate key check: country_code + year + indicator_code
dups = long_df.duplicated(subset=["country_code", "year", "indicator_code"]).sum()
print("Duplicate keys:", dups)

Duplicate keys: 0


In [14]:
# Drop duplicates if any
if dups > 0:
    long_df = long_df.drop_duplicates(subset=["country_code", "year", "indicator_code"])
    print("Duplicates dropped. New shape:", long_df.shape)

In [15]:
# Data Quality Checks: Impossible Values
# GDP growth should be in a sensible range (-50, 50) for most countries.
gdp = long_df[long_df["indicator_code"] == "NY.GDP.MKTP.KD.ZG"]
extreme_gdp = gdp[(gdp["value"] < -50) | (gdp["value"] > 50)]
print("Extreme GDP growth values:")
print(extreme_gdp[["country_name", "year", "value"]].to_string(index=False))

Extreme GDP growth values:
Empty DataFrame
Columns: [country_name, year, value]
Index: []


In [16]:
# Data Quality Checks: Coverage Gaps
# Countries with fewer than 10 observed years for any indicator
coverage = (
    long_df[long_df["value_status"] == "observed"]
    .groupby(["country_code", "indicator_code"])["year"]
    .nunique()
    .reset_index(name="n_years")
)

sparse = coverage[coverage["n_years"] < 10]
print("Sparse country-indicator series (< 10 years):")
print(sparse.sort_values(["country_code", "indicator_code"]).to_string(index=False))

Sparse country-indicator series (< 10 years):
country_code       indicator_code  n_years
         ERI BX.KLT.DINV.WD.GD.ZS        6
         ERI       NE.CON.GOVT.ZS        6
         ERI       NE.GDI.TOTL.ZS        6
         ERI       NE.TRD.GNFS.ZS        6
         ERI    NY.GDP.MKTP.KD.ZG        6
         ERI       NY.GDP.PCAP.KD        6
         MWI       NE.CON.GOVT.ZS        9
         MWI       NE.GDI.TOTL.ZS        9
         MWI       NE.TRD.GNFS.ZS        9
         SSD BX.KLT.DINV.WD.GD.ZS        4
         SSD       NE.GDI.TOTL.ZS        8
         SSD       NE.TRD.GNFS.ZS        8
         SSD    NY.GDP.MKTP.KD.ZG        7
         SSD       NY.GDP.PCAP.KD        8


In [17]:
#Summary Statistics
summary = (
    long_df[long_df["value_status"] == "observed"]
    .groupby(["indicator_code", "indicator_name"])["value"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

print(summary.to_string(index=False))

      indicator_code                                              indicator_name  count        mean         std        min          max
BX.KLT.DINV.WD.GD.ZS           Foreign direct investment, net inflows (% of GDP)    890    4.418186    7.805260 -17.292117   103.337387
      FP.CPI.TOTL.ZG                       Inflation, consumer prices (annual %)    888   10.025686   29.794584 -16.859691   557.201817
      NE.CON.GOVT.ZS General government final consumption expenditure (% of GDP)    833   15.323318    6.880989   2.048438    43.482316
      NE.GDI.TOTL.ZS                          Gross capital formation (% of GDP)    830   23.474883    9.404066  -3.679206    76.782325
      NE.TRD.GNFS.ZS                                            Trade (% of GDP)    836   69.577857   34.565790   1.995412   222.178255
   NY.GDP.MKTP.KD.ZG                                       GDP growth (annual %)    933    3.874681    4.811597 -46.082122    19.675408
      NY.GDP.PCAP.KD                          GD

In [18]:
#Country List (Geographic Scope)
countries = sorted(long_df["country_name"].unique())
print(f"Number of countries: {len(countries)}")
for c in countries:
    print(" -", c)

Number of countries: 48
 - Angola
 - Benin
 - Botswana
 - Burkina Faso
 - Burundi
 - Cabo Verde
 - Cameroon
 - Central African Republic
 - Chad
 - Comoros
 - Congo, Dem. Rep.
 - Congo, Rep.
 - Cote d'Ivoire
 - Equatorial Guinea
 - Eritrea
 - Eswatini
 - Ethiopia
 - Gabon
 - Gambia, The
 - Ghana
 - Guinea
 - Guinea-Bissau
 - Kenya
 - Lesotho
 - Liberia
 - Madagascar
 - Malawi
 - Mali
 - Mauritania
 - Mauritius
 - Mozambique
 - Namibia
 - Niger
 - Nigeria
 - Rwanda
 - Sao Tome and Principe
 - Senegal
 - Seychelles
 - Sierra Leone
 - Somalia, Fed. Rep.
 - South Africa
 - South Sudan
 - Sudan
 - Tanzania
 - Togo
 - Uganda
 - Zambia
 - Zimbabwe


In [19]:
#Sort and Finalise Column Order
final_cols = [
    "country_name", "country_code", "year",
    "indicator_code", "indicator_name",
    "value", "unit", "price_basis",
    "value_status", "observation_status",
    "periodicity", "source_dataset", "retrieval_date",
]

long_df = long_df[final_cols].sort_values(
    ["country_code", "year", "indicator_code"]
).reset_index(drop=True)

print("Final shape:", long_df.shape)
long_df.head(10)

Final shape: (7680, 13)


,country_name,country_code,year,indicator_code,indicator_name,value,unit,price_basis,value_status,observation_status,periodicity,source_dataset,retrieval_date
0,Angola,AGO,2006,BX.KLT.DINV.WD.GD.ZS,"Foreign direct investment, net inflows (% of GDP)",-0.064301,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
1,Angola,AGO,2006,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",13.305210,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
2,Angola,AGO,2006,NE.CON.GOVT.ZS,General government final consumption expenditu...,16.547943,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
3,Angola,AGO,2006,NE.GDI.TOTL.ZS,Gross capital formation (% of GDP),24.128522,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
4,Angola,AGO,2006,NE.TRD.GNFS.ZS,Trade (% of GDP),84.624604,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
5,Angola,AGO,2006,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),11.841818,percent,constant price,observed,reported,Annual,World Development Indicators,2026-07-13
6,Angola,AGO,2006,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),3065.055762,constant 2015 US$,constant price,observed,reported,Annual,World Development Indicators,2026-07-13
7,Angola,AGO,2006,SL.UEM.TOTL.ZS,"Unemployment, total (% of labour force, modell...",16.123000,percent,modelled,observed,modelled,Annual,World Development Indicators,2026-07-13
8,Angola,AGO,2007,BX.KLT.DINV.WD.GD.ZS,"Foreign direct investment, net inflows (% of GDP)",-1.223123,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
9,Angola,AGO,2007,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",12.251497,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13


In [20]:
# Export Curated Dataset to CSV
long_df.to_csv(OUT_PATH, index=False, encoding="utf-8")
print(f"Curated dataset saved to: {OUT_PATH}")
print(f"Rows: {len(long_df):,}  |  Columns: {len(long_df.columns)}")

Curated dataset saved to: data/processed/curated_dataset.csv
Rows: 7,680  |  Columns: 13


In [21]:
#  Sanity Check: Reload the CSV
check = pd.read_csv(OUT_PATH)

print("Reloaded shape:", check.shape)
print("Missing values by column:")
print(check.isna().sum())
print("\nFirst 5 rows:")
check.head()

Reloaded shape: (7680, 13)
Missing values by column:
country_name            0
country_code            0
year                    0
indicator_code          0
indicator_name          0
value                 596
unit                    0
price_basis             0
value_status            0
observation_status      0
periodicity             0
source_dataset          0
retrieval_date          0
dtype: int64

First 5 rows:


,country_name,country_code,year,indicator_code,indicator_name,value,unit,price_basis,value_status,observation_status,periodicity,source_dataset,retrieval_date
0,Angola,AGO,2006,BX.KLT.DINV.WD.GD.ZS,"Foreign direct investment, net inflows (% of GDP)",-0.064301,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
1,Angola,AGO,2006,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",13.305210,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
2,Angola,AGO,2006,NE.CON.GOVT.ZS,General government final consumption expenditu...,16.547943,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
3,Angola,AGO,2006,NE.GDI.TOTL.ZS,Gross capital formation (% of GDP),24.128522,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
4,Angola,AGO,2006,NE.TRD.GNFS.ZS,Trade (% of GDP),84.624604,percent,current price,observed,reported,Annual,World Development Indicators,2026-07-13
